In [ ]:
import pandas as pd
%run ../utils/calculations.py

# Convert CSV files into dataframes
sampled_df = pd.read_csv('../data/discontinued/discontinued-pairwise.csv')
product_df = pd.read_csv('../data/intermediary/product-info.csv')

In [3]:
def compute_hybrid_substitution_score(product_df, pairwise_df, output_csv=None):
    if output_csv:
        with open(output_csv, "w") as f:
            f.write("product_id,substitute_id,score,rank\n")

    pairwise_df_i = pairwise_df.groupby("product_i")
    pairwise_df_j = pairwise_df.groupby("product_j")

    for row in product_df.itertuples(index=False):
        product_id = row.product_id

        product_probs = pairwise_df_i.get_group(product_id) if product_id in pairwise_df_i.groups else pd.DataFrame()
        product_j_probs = pairwise_df_j.get_group(product_id) if product_id in pairwise_df_j.groups else pd.DataFrame()

        # Reverse relationships
        products_rev = product_j_probs.rename(columns={
            'product_i': 'product_j',
            'product_j': 'product_i',
            'P_i': 'P_j',
            'P_j': 'P_i'
        })

        product_probs_df = pd.concat([product_probs, products_rev], ignore_index=True)
        if product_probs_df.empty:
            continue

        # Compute metrics
        product_probs_df['jaccard'] = product_probs_df.apply(lambda x: x.P_ij / (x.P_i + x.P_j - x.P_ij), axis=1)
        product_probs_df['conditional'] = product_probs_df.apply(lambda x: ((x.P_ij / x.P_i) + (x.P_ij / x.P_j)) / 2, axis=1)
        product_probs_df['substitution_index'] = product_probs_df.apply(lambda x: ((x.P_i * x.P_j) - x.P_ij) / (x.P_i * x.P_j), axis=1)

        # Normalize
        for col in ['jaccard', 'conditional', 'substitution_index']:
            product_probs_df[col] = (product_probs_df[col] - product_probs_df[col].min()) / (product_probs_df[col].max() - product_probs_df[col].min())

        # Weighted hybrid score
        product_probs_df['score'] = (
            0.5 * product_probs_df['substitution_index'] +
            0.3 * product_probs_df['jaccard'] +
            0.2 * product_probs_df['conditional']
        )

        # Rank
        product_probs_df['rank'] = product_probs_df['score'].rank(method='dense', ascending=False)
        product_probs_df = product_probs_df.dropna(subset=['score'])
        product_probs_df['rank'] = product_probs_df['score'].rank(method='dense', ascending=False).astype(int)
        
        # Keep only top 20
        top_substitutes_df = product_probs_df[product_probs_df['rank'] <= 20].copy()

        # Select and rename columns before saving
        top_substitutes_df = top_substitutes_df[['product_i', 'product_j', 'score', 'rank']]
        top_substitutes_df.columns = ['product_id', 'substitute_id', 'score', 'rank']

        # Append to CSV
        if output_csv:
            top_substitutes_df.to_csv(output_csv, mode='a', index=False, header=False)
        else:
            return top_substitutes_df  # For testing

    print(f"Completed substitute calculations. Saved to {output_csv if output_csv else 'DataFrame'}")

compute_hybrid_substitution_score(product_df, sampled_df, output_csv="../data/discontinued/obj1/discontinued-substitutes.csv")


Completed substitute calculations. Saved to ../data/discontinued/obj1/discontinued-substitutes.csv


In [ ]:
# Category validation

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def validate_substitutes(subs_df, product_info_df):

    # Map product_id to department and aisle
    dept_map = product_info_df.set_index('product_id')['department'].to_dict()
    aisle_map = product_info_df.set_index('product_id')['aisle'].to_dict()

    # Only evaluate the top 5 substitutes
    subs_df = subs_df[subs_df['rank'] <= 5].copy()

    # Add department and aisle info to subs_df
    subs_df['product_dept'] = subs_df['product_id'].map(dept_map)
    subs_df['substitute_dept'] = subs_df['substitute_id'].map(dept_map)
    subs_df['product_aisle'] = subs_df['product_id'].map(aisle_map)
    subs_df['substitute_aisle'] = subs_df['substitute_id'].map(aisle_map)

    # Check matches
    subs_df['same_department'] = subs_df['product_dept'] == subs_df['substitute_dept']
    subs_df['same_aisle'] = subs_df['product_aisle'] == subs_df['substitute_aisle']
    subs_df['valid_substitution'] = subs_df['same_department'] & subs_df['same_aisle']
    subs_df['possible_substitution'] = subs_df['same_department'] | subs_df['same_aisle']

    # Compute base true and predicted labels
    y_true = [1] * len(subs_df)  
    dept_pred = subs_df['same_department'].astype(int)
    aisle_pred = subs_df['same_aisle'].astype(int)
    overall_pred = subs_df['valid_substitution'].astype(int)

    # Compute metrics by level
    results = {
        'department_metrics': compute_metrics(dept_pred),
        'aisle_metrics': compute_metrics(aisle_pred),
        'combined_metrics': compute_metrics(overall_pred),
    }

    # Add extra breakdown
    breakdown = subs_df.groupby(['same_department', 'same_aisle']).size().reset_index(name='count')
    results['breakdown'] = breakdown

    return results, subs_df


def validate_substitutes_by_department(subs_df, product_info_df, top_n=5):

    # Map product_id to department and aisle
    dept_map = product_info_df.set_index('product_id')['department'].to_dict()
    aisle_map = product_info_df.set_index('product_id')['aisle'].to_dict()

    # Filter to top N substitutes
    subs_df = subs_df[subs_df['rank'] <= top_n].copy()

    # Add department and aisle info
    subs_df['product_dept'] = subs_df['product_id'].map(dept_map)
    subs_df['substitute_dept'] = subs_df['substitute_id'].map(dept_map)
    subs_df['product_aisle'] = subs_df['product_id'].map(aisle_map)
    subs_df['substitute_aisle'] = subs_df['substitute_id'].map(aisle_map)

    # Check matches
    subs_df['same_department'] = subs_df['product_dept'] == subs_df['substitute_dept']
    subs_df['same_aisle'] = subs_df['product_aisle'] == subs_df['substitute_aisle']
    subs_df['valid_substitution'] = subs_df['same_department'] & subs_df['same_aisle']

    # Compute metrics per department
    dept_metrics = {}
    all_metrics_records = []
    for dept, group in subs_df.groupby('product_dept'):
        dept_pred = group['same_department'].astype(int)
        aisle_pred = group['same_aisle'].astype(int)
        combined_pred = group['valid_substitution'].astype(int)

        metrics = {
            'department_metrics': compute_metrics(dept_pred),
            'aisle_metrics': compute_metrics(aisle_pred),
            'combined_metrics': compute_metrics(combined_pred)
        }
        dept_metrics[dept] = metrics

        # Flatten metrics for summary
        for level, vals in metrics.items():
            record = {'department': dept, 'level': level}
            record.update(vals)
            all_metrics_records.append(record)

    # Summary across departments
    summary_metrics = pd.DataFrame(all_metrics_records)

    # Select only numeric columns for aggregation
    numeric_cols = summary_metrics.select_dtypes(include=np.number).columns

    # Aggregate numeric columns per level
    summary_metrics = summary_metrics.groupby('level')[numeric_cols].agg(['mean', 'std', 'min', 'max'])

    return dept_metrics, summary_metrics, subs_df



substitutes_df = pd.read_csv("../data/discontinued/obj1/discontinued-substitutes.csv")

print(substitutes_df.describe())

results, validated_df = validate_substitutes(substitutes_df, product_df)

validated_df.to_csv("../data/discontinued/obj1/discontinued-results.csv", index=False)

print("Department-level metrics:")
print(results['department_metrics'])

print("\nAisle-level metrics:")
print(results['aisle_metrics'])

print("\nCombined (Dept + Aisle) metrics:")
print(results['combined_metrics'])

print("\nBreakdown:")
print(results['breakdown'])


dept_metrics, summary_metrics, dept_validated_df = validate_substitutes_by_department(substitutes_df, product_df, top_n=5)

dept_validated_df.to_csv("../data/discontinued/obj1/discontinued-dept-results.csv", index=False)

print("=== Per-Department Metrics ===")
for dept, metrics in dept_metrics.items():
    print(f"\nDepartment: {dept}")
    print("  Department-level metrics:", metrics['department_metrics'])
    print("  Aisle-level metrics:", metrics['aisle_metrics'])
    print("  Combined (Dept + Aisle) metrics:", metrics['combined_metrics'])

print("\n=== Summary Metrics Across Departments ===")
print(summary_metrics)

print("\n=== Sample of Validated Substitutes ===")
print(validated_df.head())



          product_id  substitute_id          score           rank
count  454872.000000  454872.000000  454872.000000  454872.000000
mean    24827.649653   24582.302716       0.501782       8.486049
std     14316.414214   14463.483057       0.073202       5.797256
min         1.000000       3.000000       0.000000       1.000000
25%     12471.000000   12001.000000       0.471849       3.000000
50%     24846.000000   24183.000000       0.499454       7.000000
75%     37119.000000   36982.000000       0.533547      13.000000
max     49688.000000   49688.000000       1.000000      20.000000
Department-level metrics:
{'accuracy': 0.21080975282406889, 'precision': 1.0, 'recall': 0.21080975282406889, 'f1_score': 0.34821284240960293, 'valid_pairs': 37697, 'total_pairs': 178820}

Aisle-level metrics:
{'accuracy': 0.09581702270439549, 'precision': 1.0, 'recall': 0.09581702270439549, 'f1_score': 0.1748777774375619, 'valid_pairs': 17134, 'total_pairs': 178820}

Combined (Dept + Aisle) metrics:
{'a

In [7]:
# Save valid substitutes

valid_substitutes_df = validated_df[validated_df['valid_substitution'] == True]

valid_substitutes_df.to_csv("../data/discontinued/obj1/discontinued-valid-substitutes.csv", index=False)

In [ ]:
# Is in test set

import pandas as pd

def simple_validate_substitutes(subs_df, train_df):
    train_products = set(train_df['product_id'].unique())

    subs_df = subs_df.copy()
    subs_df['in_test_set'] = subs_df['substitute_id'].apply(lambda x: x in train_products)

    return subs_df

subs_df = pd.read_csv("../data/discontinued/obj1/discontinued-valid-substitutes.csv")  
train_df = pd.read_csv("../dataset/order_products__train.csv")

validation_df = simple_validate_substitutes(subs_df, train_df)
print(validation_df)

sub_in_test = (validation_df["in_test_set"] == True).sum()
print(sub_in_test)


       product_id  substitute_id     score  rank   product_dept  \
0               1           4770  0.484582     5         snacks   
1               3          19577  0.409192     5      beverages   
2              12          36742  0.500000     2         frozen   
3              22           2723  0.456059     4  personal care   
4              25          16246  0.532194     5         snacks   
...           ...            ...       ...   ...            ...   
17129       49677          46258  0.606909     1   canned goods   
17130       49685          44578  0.666290     2         frozen   
17131       49685          24084  0.803781     1         frozen   
17132       49688          28575  0.557208     1  personal care   
17133       49688          16071  0.265469     5  personal care   

      substitute_dept            product_aisle         substitute_aisle  \
0              snacks            cookies cakes            cookies cakes   
1           beverages                      te

In [ ]:
# Behaviour validation

import pandas as pd
import numpy as np

def validate_substitutes(subs_df, prior_df, train_df):

    # Build list of predicted substitutes
    subs_per_product = subs_df.groupby('product_id')['substitute_id'].apply(list).to_dict()

    # Build list of users who bought it in prior
    users_per_product = prior_df.groupby('product_id')['user_id'].unique().to_dict()

    # Built list of products in users' train order
    train_user_products = train_df.groupby('user_id')['product_id'].apply(set).to_dict()

    results = []
    matched_records = []

    for product_id, predicted_subs in subs_per_product.items():
        users = users_per_product.get(product_id, [])
        if len(users) == 0:
            continue

        match_count = 0

        for u in users:
            user_products = train_user_products.get(u, set())
            # Find which predicted substitutes the user bought
            bought_subs = [sub for sub in predicted_subs if sub in user_products]
            if bought_subs:
                match_count += 1
                # Record each individual match
                for sub in bought_subs:
                    matched_records.append({
                        'product_id': product_id,
                        'substitute_id': sub,
                        'user_id': u
                    })

        match_rate = match_count / len(users)

        results.append({
            'product_id': product_id,
            'num_users': len(users),
            'predicted_subs': predicted_subs,
            'match_rate': match_rate,
            'match_counts': match_count
        })

    validation_df = pd.DataFrame(results)
    matched_users_df = pd.DataFrame(matched_records)

    summary = {
        'n_products_validated': len(validation_df),
        'mean_match_rate': validation_df['match_rate'].mean(),
        'weighted_match_rate': np.average(validation_df['match_rate'], weights=validation_df['num_users']),
        'total_matches': validation_df['match_counts'].sum(),
    }

    return validation_df, matched_users_df, summary

subs_df = pd.read_csv("../data/discontinued/obj1/discontinued-valid-substitutes.csv") 
prior_df = pd.read_csv("../data/intermediary/order-products-abbrev.csv")
train_df = pd.read_csv("../data/intermediary/order-products-test.csv")

validation_df, matched_users_df, summary = validate_substitutes(
    subs_df=subs_df,
    prior_df=prior_df,
    train_df=train_df
)

validation_df.to_csv("../data/discontinued/obj1/discontinued-valid-substitutes-results-product.csv", index=False)
matched_users_df.to_csv("../data/discontinued/obj1/discontinued-valid-substitutes-results-brief.csv", index=False)

print(summary)
print(validation_df.head())

{'n_products_validated': 11813, 'mean_match_rate': 0.002040756859936193, 'weighted_match_rate': 0.000177185297590427, 'total_matches': 482}
   product_id  num_users predicted_subs  match_rate  match_counts
0           1        716         [4770]         0.0             0
1           3         74        [19577]         0.0             0
2          12        120        [36742]         0.0             0
3          22         24         [2723]         0.0             0
4          25        775        [16246]         0.0             0
